# 1. Understand and qualify synthetic data, then adapt it
Run this notebook after installing the package with `pip install -e ".[dev]"`.
It generates data if needed, inspects native measurements, and applies the canonical
mapping **only at the end**. Raw measurements are not engineered model features;
notebook 02 explores CoV, autocorrelation, CUSUM, entropy, acceleration and slope.

Full-data checks below concern structure only. Detailed statistics and fault plots
use development data; final-test fault characteristics remain hidden. Passing these
checks establishes internal consistency, not realism at a particular operator.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

# Resolve relative configured output paths consistently from any notebook.
import os

os.chdir(ROOT)
from optical_anomaly.pipeline import prepare

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    start + pd.Timedelta(days=settings["generator"]["days"] * f)
    for f in settings["splits"]
]
import numpy as np
from optical_anomaly.generator import GeneratorConfig
from optical_anomaly.optics import validate_generated
from optical_anomaly.diagnostics import (
    missingness_report, healthy_statistics, development_faults, fault_contrasts,
)
from optical_anomaly.sources import SYNTHETIC_METRICS, synthetic_adapter
from optical_anomaly.adapter import CANONICAL_METRICS
from optical_anomaly.validation import require_downstream_rx

config = GeneratorConfig(**settings["generator"])
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Reports:", REPORT.resolve())

## A. Full-dataset inventory and structural qualification
No final-test fault details are displayed. Counts describe the dataset, not model performance.

In [ ]:
native = pd.read_parquet(RUN / "telemetry.parquet")
truth = pd.read_parquet(RUN / "ground_truth.parquet")
topology = pd.read_parquet(RUN / "topology.parquet")
report = validate_generated(native, truth, topology)
step = pd.Timedelta(minutes=config.interval_minutes)
expected_rows = int(config.days * 1440 / config.interval_minutes)
groups = native.groupby("device", sort=True)
checks = dict(report["checks"])
checks["timestamps_present"] = bool(native.time.notna().all())
checks["configured_entity_count"] = native.device.nunique() == config.entities
checks["configured_rows_per_entity"] = bool(groups.size().eq(expected_rows).all())
checks["ordered_fixed_cadence"] = all(
    group.time.diff().dropna().eq(step).all()
    and group.time.iloc[0] == start
    for _, group in groups
)
visible = truth.dropna(subset=["observable_onset_time"])
checks["observable_onset_inside_fault"] = bool(
    (visible.observable_onset_time.ge(visible.onset_time)
     & visible.observable_onset_time.lt(visible.end_time)).all()
)
checks["unique_fault_ids"] = not truth.fault_id.duplicated().any()
checks = {name: bool(value) for name, value in checks.items()}
display(pd.Series({
    "rows": len(native), "ONTs": native.device.nunique(),
    "measurements": len(SYNTHETIC_METRICS), "days": config.days,
    "interval_minutes": config.interval_minutes,
    "splitters": topology.splitter_id.nunique(),
    "ports": topology.pon_port_id.nunique(), "OLTs": topology.olt_id.nunique(),
}))
display(pd.Series(checks, name="structural_pass"))
(REPORT / "structural_checks.json").write_text(json.dumps(checks, indent=2))
assert all(checks.values()), "Fix structural failures before adapting data"
# Drop final data from subsequent inspection.
native = native.loc[native.time < boundaries[2]].copy()
truth = truth.loc[truth.onset_time < boundaries[2]].copy()
healthy = native.loc[native.time < boundaries[0]].copy()

## B. Measurement dictionary and topology
OLT Tx and temperature are shared port observations repeated per ONT. Do not treat those rows as independent port measurements. FEC counts describe the preceding interval; its first row is intentionally missing.

In [ ]:
dictionary = pd.DataFrame([
    {"source": source, "canonical": name, "unit": CANONICAL_METRICS[name].unit,
     "kind": CANONICAL_METRICS[name].kind,
     "meaning": CANONICAL_METRICS[name].description,
     "used_by_current_detector": name == "rx_power_dbm"}
    for source, name in SYNTHETIC_METRICS.items()
])
display(dictionary)
dictionary.to_csv(REPORT / "measurement_dictionary.csv", index=False)
display(topology.head())
display(topology.groupby(["olt_id", "pon_port_id"]).agg(
    ONTs=("entity_id", "size"), splitters=("splitter_id", "nunique")
))

## C. Counts, missingness, gaps and distributions — development only
Longest gaps below are missing polls on the verified fixed grid. Constant counters can be legitimate when no errors occur; constant optical power deserves inspection.

In [ ]:
coverage = missingness_report(native)
coverage.to_csv(REPORT / "development_missingness.csv", index=False)
display(coverage.groupby("measurement").agg(
    observed=("observed", "sum"),
    mean_missing_fraction=("missing_fraction", "mean"),
    longest_gap_intervals=("longest_missing_intervals", "max"),
    minimum_unique_values=("unique_observed_values", "min"),
))
display(coverage.sort_values("missing_fraction", ascending=False).head(20))
summary = native[list(SYNTHETIC_METRICS)].describe(
    percentiles=[0.01, 0.5, 0.99]
).T
display(summary)
summary.to_csv(REPORT / "development_distributions.csv")

## D. Healthy temporal behaviour and intended statistical properties
Remove OLT Tx from downstream Rx to isolate path loss, then remove a daily harmonic.
Expected residual variance includes physical AR(1) noise **and** independent sensor
noise. Their mixture reduces lag-1 correlation relative to physical noise alone.
Bands are screening tolerances, not formal confidence intervals or industry limits.
A warning requires investigation; do not tune the simulator simply to turn it green.

In [ ]:
statistics = healthy_statistics(healthy, config)
statistics.to_csv(REPORT / "healthy_statistics.csv", index=False)
display(statistics)
display(statistics.filter(like="in_band").mean().rename("fraction_in_band"))
entity = healthy.device.iloc[0]
view = healthy.loc[healthy.device.eq(entity)].set_index("time")
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
view[["rx_dbm", "upstream_rx_dbm"]].plot(ax=axes[0], ylabel="Rx (dBm)")
view[["ont_tx_dbm", "olt_tx_dbm"]].plot(ax=axes[1], ylabel="Tx (dBm)")
view[["ont_temperature_c", "olt_temperature_c"]].plot(ax=axes[2], ylabel="C")
plt.tight_layout()
plt.show()
(view.rx_dbm - view.rx_dbm.median()).groupby(view.index.hour).median().plot(
    title="Healthy daily profile, one ONT", ylabel="Centred Rx (dB)"
)
plt.show()

## E. Faults and dependencies — development only
Inspect duration diversity, subtle versus large changes, non-impacting faults and
warning opportunities. The opportunity column is an availability proxy, not recall.
Observable onset uses the simulator's known effect, not a field-observable truth.
BER is an assumed receiver response, and FEC uses the preceding physical sample.
Strong relationships are partly constructed; they are not independent validation.

In [ ]:
faults = development_faults(
    truth, native, boundaries[2], config.interval_minutes
)
faults.to_csv(REPORT / "development_faults.csv", index=False)
display(faults.groupby("fault_type").agg(
    faults=("fault_id", "size"), impacts=("impact_time", "count"),
    opportunities=("warning_opportunity_proxy", "sum"),
    median_duration_hours=("duration_hours", "median"),
    median_warning_minutes=("warning_minutes", "median"),
))
for fault in faults.groupby("fault_type").head(1).itertuples():
    sample = native.loc[
        native.device.eq(fault.entity_id)
        & native.time.between(
            fault.onset_time - pd.Timedelta(hours=12),
            fault.end_time + pd.Timedelta(hours=6),
        )
    ].set_index("time")
    ax = sample[["rx_dbm", "upstream_rx_dbm"]].plot(
        figsize=(11, 3), title=f"{fault.entity_id}: {fault.fault_type}",
        ylabel="Rx (dBm)",
    )
    for name in ["onset_time", "observable_onset_time", "impact_time", "end_time"]:
        when = getattr(fault, name)
        if pd.notna(when):
            ax.axvline(when, linestyle="--", label=name)
    ax.legend(fontsize=8)
    plt.show()
# One ONT avoids treating shared port rows as independent observations.
view = native.loc[native.device.eq(entity)].set_index("time")
ratios = pd.DataFrame(index=view.index)
for direction in ["downstream", "upstream"]:
    total = view[f"{direction}_fec_total_codewords"].where(lambda x: x > 0)
    for kind in ["corrected", "uncorrectable"]:
        ratios[f"{direction}_{kind}"] = (
            view[f"{direction}_fec_{kind}_codewords"] / total
        )
ratios.plot(figsize=(12, 3), ylabel="FEC fraction", title=entity)
plt.show()
display(view[["rx_dbm", "upstream_rx_dbm", "ber", "upstream_ber"]].corr(
    method="spearman"
))
contrasts = fault_contrasts(native, faults)
contrasts.to_csv(REPORT / "development_fault_contrasts.csv", index=False)
display(contrasts)
print("Inspect large drops, low severity diversity, and fault-associated missingness.")

## F. Qualification decision and remaining limitations
**Must pass:** structural checks. **Investigate:** statistical-band failures,
long gaps, constant optical measurements, implausibly easy faults, and limited
healthy exposure. No single automatic score certifies synthetic realism.

Current limitations remain explicit:
- Fault timing is concentrated in scenario windows, with enriched fault prevalence.
- Missing polls are independent of severity; outage-related censoring is absent.
- No shared splitter/port faults or empirical hardware/receiver calibration.
- BER and FEC relationships are simulated; real vendor counter semantics may differ.
- Multiple seeds and parameter stress experiments are needed before robustness claims.
- Final-test details remain excluded from these development diagnostics.

Review `METHOD.md` for standards, assumptions and evidence. Reports are written to
`RUN/eda/`; the original telemetry and truth files are not modified.

## G. Canonical adaptation — after inspection
The synthetic source has an explicit mapping, just as each company would.
Adapt **one development ONT** here to keep the 14-measurement long-table preview
manageable. The model pipeline separately selects downstream Rx for all ONTs.
Missing mapped columns are errors. Cumulative counters require explicit conversion.
The detector requirement is separate from source interpretation.

In [ ]:
preview_native = native.loc[native.device.eq(entity)]
adapter = synthetic_adapter()  # Explicit full synthetic source contract.
canonical = adapter.transform(preview_native)
display(canonical.head(20))
display(canonical.groupby("metric_name").agg(
    rows=("value", "size"), observed=("value", "count")
))
require_downstream_rx(canonical)
print("Downstream Rx is available; training history is checked during fitting.")
canonical.head(100).to_csv(REPORT / "canonical_preview.csv", index=False)